# 06 — Supplementary analyses

Analyses added during peer review. Except for the muscle-activity index, which is computed
from the raw EDF files, everything is computed from the outputs of notebooks 00–03; no
feature or model is recomputed.

| Section | Content | Main outputs |
|---|---|---|
| 1 | Windows per participant and condition (Table S1) | printed summary of `subject_window_counts.csv` |
| 2 | Spectral composition: total, absolute and relative band power (Table S2) | `S2_spectral_composition.csv`, `tabS_composicao.tex` |
| 3 | Between-condition contrasts of every feature (Table S3, Supplementary Data S1) | `S3_contrasts.csv`, `tabS_contrastes.tex`, `tabS_contrastes_full.tex` |
| 4 | Muscle-activity (EMG) index and sensitivity of the beta difference | `emg_*.csv` |
| 5 | Cliff's δ intervals adjusted for the false coverage rate (Benjamini & Yekutieli, 2005) | `fcr_intervals.csv`, `fcr_summary.csv` |
| 6 | Hedges' g of relative beta power | `hedges_g_relative_beta.csv` |
| 7 | Individual values (Fig. S1) | `figS1_individual_values.pdf/.png`, `figS1_deltas.csv` |
| 8 | Connectivity and relative theta: features cited in the Results | printed |

Controls are split by recording session as in `03_condition_dependence.ipynb`: January 2013
(`JAN`) and the controls recorded in May–July 2013, contemporaneous with the patients
(`CONTEMP`). Cliff's δ is P(a > b) − P(a < b): positive values mean higher in MDD for
group comparisons, and higher in the January controls for the batch effect.

## Setup

In [ ]:
# Google Colab only: install dependencies and mount Google Drive.
import sys
if 'google.colab' in sys.modules:
    !pip install -q mne==1.13.2 statsmodels PyWavelets
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os
import sys

REPO_DIR = '..'   # repository root (on Colab: the folder where the repository was cloned or copied)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

import re
import warnings

import matplotlib
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from scipy import signal, stats
from statsmodels.stats.multitest import multipletests

from condition_tests import CONTRASTS, EQUIV_MARGIN
from eeg_features import BANDS, CHANNEL_NAMES, CONDITIONS, REGIONS, SEED

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')
print(f'Python {sys.version.split()[0]}')
for module in (mne, np, pd, statsmodels, matplotlib):
    print(f'{module.__name__:12s} {module.__version__}')

## Configuration

In [ ]:
RESULTS_DIR = 'path/to/results'             # outputs of notebooks 02 and 03
AUDIT_DIR = 'path/to/audit_outputs'         # outputs of 00_duplicate_audit.ipynb (file_inventory.csv)
EDF_DIR = 'path/to/original_edf_files'      # original EDF files (section 4)
OUT_DIR = os.path.join(RESULTS_DIR, 'supplementary')
os.makedirs(OUT_DIR, exist_ok=True)

BATCH_SESSION = ('2013-01-01', '2013-01-31')   # as in 03_condition_dependence.ipynb
ALPHA = 0.05
SFREQ = 256
N_BOOT_FCR = 10_000            # bootstrap resamples, section 5
SENSITIVITY_MARGINS = (0.20, 0.50)
EMG_BAND = (55, 75)            # Hz
EMG_REFERENCE_BAND = (1, 45)   # Hz
TEMPORAL_CHANNELS = ['T3', 'T4', 'T5', 'T6']
SITES = CHANNEL_NAMES + list(REGIONS)
BAND_NAMES = list(BANDS)       # the five bands partition 1-45 Hz
DOMAIN_LABEL = {'spectral': 'Spectral', 'time-frequency': 'Time-frequency', 'asymmetry': 'Asymmetry',
                'ratio': 'Band ratios', 'complexity': 'Complexity', 'connectivity': 'Connectivity'}


def cliff(a, b):
    # P(a > b) - P(a < b)
    return float(np.sign(np.subtract.outer(np.asarray(a, float), np.asarray(b, float))).mean())

## Data

In [ ]:
features = {c: pd.read_csv(os.path.join(RESULTS_DIR, f'features_{c}.csv'), float_precision='round_trip')
            .set_index('subject') for c in CONDITIONS}
stats_by_cond = {c: pd.read_csv(os.path.join(RESULTS_DIR, f'stats_{c}.csv'), float_precision='round_trip')
                 .set_index('feature') for c in CONDITIONS}
for c in CONDITIONS:
    assert list(features[c].index) == list(features[CONDITIONS[0]].index), f'{c}: different participants'
subjects = features[CONDITIONS[0]].index
FEATURES = [f for f in features[CONDITIONS[0]].columns if f != 'group']

inventory = pd.read_csv(os.path.join(AUDIT_DIR, 'file_inventory.csv'), parse_dates=['start'])
first = inventory.groupby('subject_id')['start'].min()
first = first.dt.tz_localize(None) if first.dt.tz is not None else first
in_session = (first >= pd.Timestamp(BATCH_SESSION[0])) & (first < pd.Timestamp(BATCH_SESSION[1]) + pd.Timedelta(days=1))
MDD = subjects[features['EC'].group.eq('mdd')]
HC = subjects[features['EC'].group.eq('control')]
JAN = HC[in_session.reindex(HC).to_numpy()]
CONTEMP = HC.difference(JAN)
print(f'{len(HC)} HC ({len(JAN)} January 2013, {len(CONTEMP)} contemporaneous: {list(CONTEMP)}) + {len(MDD)} MDD '
      f'| {len(FEATURES)} features')

## 1. Windows per participant and condition (Table S1)

`subject_window_counts.csv` (notebook 02) holds, for every participant and condition, the
valid 2-s windows available, the windows used (at most `n_target`) and the 10-s blocks available.

In [ ]:
counts = pd.read_csv(os.path.join(RESULTS_DIR, 'subject_window_counts.csv'))
long = pd.concat([counts[['subject', 'group']].assign(
    condition=c, available=counts[f'n_windows_available_{c}'], used=counts[f'n_windows_used_{c}'],
    blocks=counts[f'n_blocks_available_{c}']) for c in CONDITIONS], ignore_index=True)
n_target = int(long.used.max())
below = long[long.available < n_target]
print(f'Participant-condition pairs: {len(long)} | n_target = {n_target} windows')
print(f'Pairs with fewer than n_target windows (all windows used): {len(below)}')
if len(below):
    print(below.sort_values('available')[['subject', 'group', 'condition', 'available']].to_string(index=False))
print(f'Smallest number of available windows: {long.available.min()} '
      f'({", ".join(long.loc[long.available == long.available.min(), "subject"] + " " + long.loc[long.available == long.available.min(), "condition"])})')
print(long.groupby('condition')[['available', 'used', 'blocks']].agg(['min', 'median', 'max']).to_string())

## 2. Spectral composition (Table S2)

For each of the 24 sites, condition and measure: total power 1–45 Hz (sum of the absolute
powers of the five bands; for a region, the mean of the channel totals), absolute beta power
and the relative power of the five bands. Mann–Whitney p-values are FDR-corrected across the
24 sites within measure and condition (descriptive).

In [ ]:
def composition_measures(df, site):
    m = {'total_1_45': sum(df[f'abs_{b}_{site}'] for b in BAND_NAMES), 'beta_abs': df[f'abs_beta_{site}']}
    m.update({f'{b}_rel': df[f'rel_{b}_{site}'] for b in BAND_NAMES})
    return m


rows = []
for cond in CONDITIONS:
    df = features[cond]
    for site in SITES:
        for name, v in composition_measures(df, site).items():
            rows.append(dict(condition=cond, site=site, measure=name,
                             median_hc=v[HC].median(), median_mdd=v[MDD].median(),
                             p=stats.mannwhitneyu(v[MDD], v[HC], alternative='two-sided').pvalue,
                             delta=cliff(v[MDD], v[HC]), delta_vs_contemp=cliff(v[MDD], v[CONTEMP]),
                             delta_batch=cliff(v[JAN], v[CONTEMP])))
composition = pd.DataFrame(rows)
composition['q_sites'] = composition.groupby(['condition', 'measure'])['p'].transform(
    lambda p: multipletests(p, method='fdr_bh')[1])
composition.to_csv(os.path.join(OUT_DIR, 'S2_spectral_composition.csv'), index=False)

MEASURES = [('total_1_45', 'Total power, 1--45 Hz'), ('beta_abs', 'Absolute beta'), ('beta_rel', 'Relative beta'),
            ('delta_rel', 'Relative delta'), ('theta_rel', 'Relative theta'), ('alpha_rel', 'Relative alpha'),
            ('gamma_rel', 'Relative gamma')]
summary = (composition.groupby(['measure', 'condition'])
           .agg(delta_median=('delta', 'median'), delta_min=('delta', 'min'), delta_max=('delta', 'max'),
                delta_contemp_median=('delta_vs_contemp', 'median'), delta_batch_median=('delta_batch', 'median'),
                sites_q05=('q_sites', lambda q: int((q < 0.05).sum())))
           .reindex([m for m, _ in MEASURES], level=0))
summary.to_csv(os.path.join(OUT_DIR, 'S2_spectral_composition_summary.csv'))
print("Cliff's δ over the 24 sites (median, range) and number of sites with q < 0.05:")
print(summary.round(2).to_string())
print('\nT4:')
print(composition[(composition.site == 'T4') & composition.measure.isin(['beta_abs', 'beta_rel', 'total_1_45'])]
      [['condition', 'measure', 'delta', 'delta_vs_contemp', 'delta_batch', 'q_sites']].round(3).to_string(index=False))

In [ ]:
def tex_number(x):
    if abs(x) < 0.005:
        return '0.00'
    return f'{x:+.2f}'.replace('-', '$-$').replace('+', '$+$')


def tex_summary(v):
    return f'{tex_number(v.median())} [{tex_number(v.min())}, {tex_number(v.max())}]'


L = [r'\begin{table*}[!t]',
     r'\caption{Spectral composition: total, absolute and relative band power, MDD vs.\ HC}',
     r'\label{tabS_composicao}', r'\centering', r'\footnotesize', r'\setlength{\tabcolsep}{4pt}',
     r'\renewcommand{\arraystretch}{1.1}', r'\begin{tabular}{llcccc}', r'\toprule',
     r'\textbf{Measure} & \textbf{Condition} & \textbf{MDD vs.\ all HC} & \textbf{Sites with} & '
     r'\textbf{MDD vs.\ contemp.\ HC} & \textbf{Batch: Jan vs.\ May--Jul HC} \\',
     r' & & \textbf{median [min, max]} & \textbf{$q < 0.05$} & \textbf{median [min, max]} & \textbf{median [min, max]} \\',
     r'\midrule']
for k, (measure, label) in enumerate(MEASURES):
    for i, cond in enumerate(CONDITIONS):
        s = composition[(composition.measure == measure) & (composition.condition == cond)]
        L.append(f"{label if i == 0 else ''} & {cond} & {tex_summary(s.delta)} & "
                 f"{int((s.q_sites < 0.05).sum())}/{len(SITES)} & {tex_summary(s.delta_vs_contemp)} & "
                 f"{tex_summary(s.delta_batch)} \\\\")
    if k == 2:
        L.append(r'\midrule')
    elif k < len(MEASURES) - 1:
        L.append(r'\addlinespace')
L += [r'\bottomrule', r'\end{tabular}', r'\par\vspace{2pt}', r'\begin{minipage}{\textwidth}', r'\scriptsize',
      r"Cliff's $\delta$ (positive: higher in MDD) summarized over the 24 sites (19 channels and 5 regions), "
      r'so that all measures are summarized over the same set; Table~\ref{tab10} instead gives the median over '
      r'the significant sites of each family. Total power is the sum of the absolute powers of the five bands, '
      r'which partition 1--45~Hz; for regions, it is the mean of the channel totals. $q$: Mann--Whitney $p$ values '
      r'FDR-corrected across the 24 sites, within measure and condition. Contemporaneous (contemp.) HC: the '
      f'controls recorded in May--July 2013, like the patients ($n = {len(CONTEMP)}$). Batch: controls '
      f'recorded in January 2013 ($n = {len(JAN)}$, all with artifact removal) vs.\\ contemporaneous controls; '
      r'positive values indicate higher values in the January controls.',
      r'\end{minipage}', r'\end{table*}']
with open(os.path.join(OUT_DIR, 'tabS_composicao.tex'), 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(L))
print('saved tabS_composicao.tex')

## 3. Between-condition contrasts (Table S3 and Supplementary Data S1)

The three contrasts of the MDD−HC difference (EO−EC, TASK−EC, TASK−EO), in SD units of the
transformed and standardized feature, as estimated in Layers 2–3 (`layer3_equivalence.csv`,
REML mixed model): estimate, SE, 90% CI (the TOST interval) and 95% CI, two-sided p with
2(N−2) df, Holm p within feature, TOST p (Δ = 0.30 SD) and the smallest margin at which the
contrast would be declared equivalent, max(|lower|, |upper|) of the 90% CI.

In [ ]:
l3 = pd.read_csv(os.path.join(RESULTS_DIR, 'layer3_equivalence.csv'), float_precision='round_trip')
order = {d: i for i, d in enumerate(DOMAIN_LABEL)}
l3 = l3.assign(_o=l3.domain.map(order)).sort_values(['_o', 'feature']).drop(columns='_o').reset_index(drop=True)
LABEL = {'EO_vs_EC': 'EO-EC', 'TASK_vs_EC': 'TASK-EC', 'TASK_vs_EO': 'TASK-EO'}
df_t = l3.df_resid.to_numpy(float)
s3 = pd.DataFrame({'feature': l3.feature, 'domain': l3.domain.map(DOMAIN_LABEL),
                   'log10': l3.transform_perm.eq('log10'), 'n_hc': l3.n_hc, 'n_mdd': l3.n_mdd, 'df': df_t,
                   'F': l3.F_interaction, 'p_F': l3.p_interaction_F, 'p_perm': l3.p_interaction_perm,
                   'q_perm': l3.p_interaction_fdr})
for c in CONTRASTS:
    n, est, se = LABEL[c], l3[f'delta_effect_{c}_z'], l3[f'se_{c}_z']
    t95 = stats.t.ppf(0.975, df_t)
    s3[f'{n}_est'], s3[f'{n}_se'] = est, se
    s3[f'{n}_ci90_low'], s3[f'{n}_ci90_high'] = l3[f'ci90_low_{c}'], l3[f'ci90_high_{c}']
    s3[f'{n}_ci95_low'], s3[f'{n}_ci95_high'] = est - t95 * se, est + t95 * se
    s3[f'{n}_p'], s3[f'{n}_p_holm'], s3[f'{n}_p_tost'] = l3[f'p_{c}'], l3[f'p_{c}_holm'], l3[f'p_tost_{c}']
    s3[f'{n}_min_margin'] = np.maximum(s3[f'{n}_ci90_low'].abs(), s3[f'{n}_ci90_high'].abs())
s3['p_tost_global'], s3['q_tost_global'] = l3.p_equivalence_global, l3.p_equivalence_fdr
s3['min_margin_global'] = s3[[f'{LABEL[c]}_min_margin' for c in CONTRASTS]].max(axis=1)
s3.to_csv(os.path.join(OUT_DIR, 'S3_contrasts.csv'), index=False)

names = [LABEL[c] for c in CONTRASTS]
nu = int(np.median(df_t))
se_med = float(np.median(s3[[f'{n}_se' for n in names]].to_numpy()))
print(f'{len(s3)} features ({int(s3.log10.sum())} log10) | df = {nu} | median SE {se_med:.3f} | '
      f'median 90% CI half-width {stats.t.ppf(0.95, nu) * se_med:.3f} | '
      f'MDE {(stats.t.ppf(0.975, nu) + stats.t.ppf(0.80, nu)) * se_med:.3f} | '
      f'equivalent (FDR, Δ = {EQUIV_MARGIN}): {int((s3.q_tost_global < ALPHA).sum())} | '
      f'smallest global margin {s3.min_margin_global.min():.2f} SD')
print('Features per domain:', s3.domain.value_counts().to_dict())

In [ ]:
def tex_signed(x, nd=2):
    s = f'{x:.{nd}f}'
    return s.replace('-', '$-$') if s.startswith('-') else s


def tex_escape(s):
    return s.replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')


rot = {'EO-EC': 'EO $-$ EC', 'TASK-EC': 'TASK $-$ EC', 'TASK-EO': 'TASK $-$ EO'}
t90 = stats.t.ppf(0.95, nu)
L = [r'\begin{table*}[!t]',
     r'\caption{Between-condition contrasts of the MDD$-$HC difference, summarized by domain}',
     r'\label{tabS_contrastes}', r'\centering', r'\footnotesize', r'\renewcommand{\arraystretch}{1.1}',
     r'\setlength{\tabcolsep}{4pt}', r'\begin{tabular}{llccccccc}', r'\toprule',
     r'\textbf{Domain} & \textbf{Contrast} & \textbf{$n$} & \textbf{Median $|\hat{c}|$} & \textbf{Max $|\hat{c}|$}'
     r' & \textbf{Median SE} & \textbf{Median 90\% CI} & \textbf{90\% CI within} & \textbf{90\% CI within} \\',
     r' & & & & & & \textbf{half-width} & \textbf{$\pm 0.30$ SD} & \textbf{$\pm 0.50$ SD} \\', r'\midrule']
domains = [d for d in DOMAIN_LABEL.values() if d in set(s3.domain)]
for d in domains + ['All features']:
    sub = s3 if d == 'All features' else s3[s3.domain == d]
    if d == 'All features':
        L.append(r'\midrule')
    for i, n in enumerate(names):
        e, se = sub[f'{n}_est'], sub[f'{n}_se']
        lo, hi = sub[f'{n}_ci90_low'], sub[f'{n}_ci90_high']
        L.append(f"{tex_escape(d) if i == 0 else ''} & {rot[n]} & {len(sub) if i == 0 else ''} & "
                 f"{tex_signed(e.abs().median())} & {tex_signed(e.abs().max())} & {tex_signed(se.median())} & "
                 f"{tex_signed(t90 * se.median())} & {int(((lo > -0.30) & (hi < 0.30)).sum())} & "
                 f"{int(((lo > -0.50) & (hi < 0.50)).sum())} \\\\")
    if d not in ('All features', domains[-1]):
        L.append(r'\addlinespace')
L += [r'\bottomrule', r'\end{tabular}', r'\par\vspace{2pt}', r'\begin{minipage}{\textwidth}', r'\scriptsize',
      r'Each contrast is the change in the MDD$-$HC difference between two conditions, e.g.\ EO $-$ EC '
      r'$= (\mathrm{MDD}-\mathrm{HC})_{\mathrm{EO}} - (\mathrm{MDD}-\mathrm{HC})_{\mathrm{EC}}$, estimated from the '
      r'linear mixed model with random intercept per participant (REML), in standard-deviation units of the '
      f'transformed and standardized feature; $t$ distribution with {nu} degrees of freedom. '
      r'The 90\% CI is the interval used by the equivalence test (TOST): a contrast is equivalent within $\pm\Delta$ '
      r'when its 90\% CI lies inside $[-\Delta, +\Delta]$. The last two columns count features whose 90\% CI lies '
      r'inside the margin before correction for multiple comparisons; global equivalence additionally requires all '
      f'three contrasts and FDR correction across features ({int((s3.q_tost_global < ALPHA).sum())} features at '
      f'$\\Delta = {EQUIV_MARGIN:.2f}$). Estimates, SE, 90\\% and 95\\% CIs, $p$ values (Holm-adjusted within '
      r'feature) and the smallest achievable margin for each of the 1,216 features are given in Supplementary Data~S1.',
      r'\end{minipage}', r'\end{table*}']
with open(os.path.join(OUT_DIR, 'tabS_contrastes.tex'), 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(L))

cell = lambda row, n: (f"{tex_signed(row[n + '_est'])} [{tex_signed(row[n + '_ci90_low'])}, "
                       f"{tex_signed(row[n + '_ci90_high'])}]")
header = (r'\textbf{Domain} & \textbf{Feature} & \textbf{EO $-$ EC} & \textbf{TASK $-$ EC} & '
          r'\textbf{TASK $-$ EO} & \textbf{Min.\ margin} \\')
L = [r'% Requires \usepackage{longtable,booktabs}; in a two-column document, use \onecolumn before \input.',
     r'{\scriptsize', r'\setlength{\tabcolsep}{3pt}', r'\begin{longtable}{llcccc}',
     r'\caption{Between-condition contrasts of the MDD$-$HC difference for each feature: '
     r'estimate [90\% CI], SD units}\label{tabS_contrastes_completa}\\',
     r'\toprule', header, r'\midrule', r'\endfirsthead', r'\toprule', header, r'\midrule', r'\endhead',
     r'\bottomrule', r'\endfoot']
for _, row in s3.iterrows():
    L.append(f'{tex_escape(row.domain)} & \\texttt{{{tex_escape(row.feature)}}} & '
             + ' & '.join(cell(row, n) for n in names) + f' & {tex_signed(row.min_margin_global)} \\\\')
L += [r'\end{longtable}', '}']
with open(os.path.join(OUT_DIR, 'tabS_contrastes_full.tex'), 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(L))
print('saved tabS_contrastes.tex, tabS_contrastes_full.tex')

## 4. Muscle-activity (EMG) index

From the raw EDF file of every record of the analytic sample (before ICA): 19 channels,
average reference, 50-Hz notch, 1-Hz high-pass, non-overlapping 2-s windows, Welch PSD
(256-sample Hann segments, 50% overlap). Power in 55–75 Hz = median PSD in the band × number
of bins (robust to narrow peaks); relative index = log10[P(55–75) / P(1–45)]; absolute index =
log10 P(55–75); median over windows, then mean over channels (global) or over T3–T6
(temporal). Beta is related to the index within each group (Spearman), and the group effect on
the rank of beta is estimated with and without the rank of the index as covariate.

In [ ]:
def edf_index():
    index = {}
    for name in os.listdir(EDF_DIR):
        m = re.match(r'^(H|MDD)\s*S\s*(\d+)\s+(EC|EO|TASK)\.edf$', re.sub(r'\s+', ' ', name.strip()), flags=re.I)
        if m:
            index[(f"{'HC' if m.group(1).upper() == 'H' else 'MDD'}_{int(m.group(2))}", m.group(3).upper())] = name
    return index


def emg_index(path):
    raw = mne.io.read_raw_edf(path, preload=True, verbose='error')
    mapping = {}
    for c in CHANNEL_NAMES:
        found = [n for n in raw.ch_names if re.search(rf'(^|[^A-Za-z]){re.escape(c)}([^A-Za-z0-9]|$)', n, flags=re.I)]
        assert len(found) == 1, f'channel {c}: {found}'
        mapping[found[0]] = c
    raw.pick(list(mapping)).rename_channels(mapping).reorder_channels(CHANNEL_NAMES)
    raw.set_eeg_reference('average', verbose='error')
    raw.notch_filter([50.0], verbose='error')
    raw.filter(l_freq=1.0, h_freq=None, verbose='error')
    x = raw.get_data() * 1e6
    n_win = 2 * SFREQ
    n = x.shape[1] // n_win
    windows = x[:, :n * n_win].reshape(len(CHANNEL_NAMES), n, n_win)
    f, pxx = signal.welch(windows, fs=SFREQ, window='hann', nperseg=SFREQ, noverlap=SFREQ // 2,
                          detrend='constant', axis=-1)
    m_emg = (f >= EMG_BAND[0]) & (f <= EMG_BAND[1])
    m_ref = (f >= EMG_REFERENCE_BAND[0]) & (f <= EMG_REFERENCE_BAND[1])
    p_emg = np.median(pxx[..., m_emg], axis=-1) * m_emg.sum()
    log_rel = np.median(np.log10(p_emg / pxx[..., m_ref].sum(axis=-1)), axis=1)
    log_abs = np.median(np.log10(p_emg), axis=1)
    it = [CHANNEL_NAMES.index(c) for c in TEMPORAL_CHANNELS]
    return dict(emg_rel_global=log_rel.mean(), emg_rel_temporal=log_rel[it].mean(),
                emg_rel_T4=log_rel[CHANNEL_NAMES.index('T4')], emg_abs_global=log_abs.mean(),
                emg_abs_temporal=log_abs[it].mean(), n_windows=n)


edfs = edf_index()
missing = [(s, c) for c in CONDITIONS for s in subjects if (s, c) not in edfs]
assert not missing, f'records without EDF: {missing[:10]}'
emg = pd.DataFrame([dict(subject=s, condition=c, group=features[c].loc[s, 'group'],
                         **emg_index(os.path.join(EDF_DIR, edfs[(s, c)])))
                    for c in CONDITIONS for s in subjects])
emg.to_csv(os.path.join(OUT_DIR, 'emg_index.csv'), index=False)
print(f'EMG index computed for {len(emg)} records')

In [ ]:
def rank_group_effect(y, is_mdd, covariate=None):
    # OLS on standardized ranks: rank(y) ~ group [+ rank(covariate)]; group coefficient in SD of the rank.
    z = lambda v: (stats.rankdata(v) - stats.rankdata(v).mean()) / stats.rankdata(v).std(ddof=1)
    X = [is_mdd.astype(float)] + ([z(covariate)] if covariate is not None else [])
    fit = sm.OLS(z(y), sm.add_constant(np.column_stack(X))).fit()
    return fit.params[1], fit.pvalues[1]


def beta(df, kind, target):
    if target == 'mean19':
        return df[[f'{kind}_beta_{c}' for c in CHANNEL_NAMES]].mean(axis=1)
    return df[f'{kind}_beta_{target}']


INDICES = ['emg_rel_global', 'emg_rel_temporal', 'emg_rel_T4', 'emg_abs_global', 'emg_abs_temporal']
comparison, correlation, sensitivity = [], [], []
for cond in CONDITIONS:
    df = features[cond]
    e = emg[emg.condition == cond].set_index('subject').loc[df.index]
    is_mdd = df.group.eq('mdd').to_numpy()
    for ind in INDICES:
        v = e[ind]
        comparison.append(dict(condition=cond, index=ind, median_hc=v[HC].median(), median_mdd=v[MDD].median(),
                               p=stats.mannwhitneyu(v[MDD], v[HC], alternative='two-sided').pvalue,
                               delta=cliff(v[MDD], v[HC]), delta_vs_contemp=cliff(v[MDD], v[CONTEMP])))
    for kind in ('rel', 'abs'):
        for target in ('mean19', 'T4'):
            y = beta(df, kind, target)
            for ind in ('emg_rel_global', 'emg_rel_temporal'):
                for g, ids in (('HC', HC), ('MDD', MDD)):
                    rho, p = stats.spearmanr(y[ids], e.loc[ids, ind])
                    correlation.append(dict(condition=cond, beta=f'{kind}_{target}', index=ind, group=g, rho=rho, p=p))
                b0, p0 = rank_group_effect(y.to_numpy(), is_mdd)
                b1, p1 = rank_group_effect(y.to_numpy(), is_mdd, e[ind].to_numpy())
                sensitivity.append(dict(condition=cond, beta=f'{kind}_{target}', index=ind,
                                        group_coef_without=b0, p_without=p0, group_coef_with=b1, p_with=p1))
comparison, correlation, sensitivity = map(pd.DataFrame, (comparison, correlation, sensitivity))
main = comparison['index'] != 'emg_abs_temporal'   # FDR over the four main indices; temporal absolute is descriptive
comparison['q'] = np.nan
comparison.loc[main, 'q'] = multipletests(comparison.loc[main, 'p'], method='fdr_bh')[1]
correlation['q'] = multipletests(correlation.p, method='fdr_bh')[1]
for name, t in (('emg_group_comparison', comparison), ('emg_beta_correlation', correlation),
                ('emg_rank_sensitivity', sensitivity)):
    t.to_csv(os.path.join(OUT_DIR, f'{name}.csv'), index=False)

print('EMG index between groups:')
print(comparison.round(3).to_string(index=False))
sig = correlation[correlation.q < 0.10]
print('\nSpearman beta x EMG index within groups, q < 0.10:')
print(sig.round(3).to_string(index=False) if len(sig) else '  none')
print(f'  rho: median {correlation.rho.median():.2f}, range {correlation.rho.min():.2f} to {correlation.rho.max():.2f}')
print('\nGroup effect on the rank of beta, without and with the EMG index:')
print(sensitivity.round(3).to_string(index=False))

## 5. Cliff's δ intervals adjusted for the false coverage rate

For the R features selected by FDR in each condition (out of m = 1,216), percentile bootstrap
intervals (`N_BOOT_FCR` resamples of participants within each group) at level 1 − Rα/m
(Benjamini & Yekutieli, 2005), together with the unadjusted 95% interval.

In [ ]:
rng = np.random.default_rng(SEED)
rows, summary = [], []
for cond in CONDITIONS:
    df = features[cond]
    Xm, Xh = df.loc[MDD, FEATURES].to_numpy(float), df.loc[HC, FEATURES].to_numpy(float)
    selected = stats_by_cond[cond].reindex(FEATURES).significant.to_numpy(bool)
    R, m = int(selected.sum()), len(FEATURES)
    level = 1 - R * ALPHA / m
    a = (1 - level) / 2
    im = rng.integers(0, Xm.shape[0], (N_BOOT_FCR, Xm.shape[0]))
    ih = rng.integers(0, Xh.shape[0], (N_BOOT_FCR, Xh.shape[0]))
    for j in np.flatnonzero(selected):
        S = np.sign(np.subtract.outer(Xm[:, j], Xh[:, j]))
        boot = S[im[:, :, None], ih[:, None, :]].mean(axis=(1, 2))
        lo_f, hi_f = np.quantile(boot, [a, 1 - a])
        lo_95, hi_95 = np.quantile(boot, [0.025, 0.975])
        rows.append(dict(condition=cond, feature=FEATURES[j], delta=S.mean(), ci95_low=lo_95, ci95_high=hi_95,
                         ci_fcr_low=lo_f, ci_fcr_high=hi_f, fcr_excludes_zero=(lo_f > 0) or (hi_f < 0)))
    t = pd.DataFrame([r for r in rows if r['condition'] == cond])
    lower_abs = np.where(t.delta > 0, t.ci_fcr_low, -t.ci_fcr_high)
    summary.append(dict(condition=cond, R=R, fcr_level_pct=round(100 * level, 2),
                        excluding_zero=int(t.fcr_excludes_zero.sum()),
                        median_lower_limit_abs_delta=round(float(np.median(lower_abs)), 3)))
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, 'fcr_intervals.csv'), index=False)
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT_DIR, 'fcr_summary.csv'), index=False)
print(summary.to_string(index=False))

## 6. Hedges' g of relative beta power (from `stats_<COND>.csv`)

In [ ]:
g = pd.concat([stats_by_cond[c].loc[[f'rel_beta_{s}' for s in SITES], ['hedges_g', 'cliffs_delta']]
               .assign(condition=c, site=SITES) for c in CONDITIONS]).reset_index()
g.to_csv(os.path.join(OUT_DIR, 'hedges_g_relative_beta.csv'), index=False)
print(g.groupby('condition').hedges_g.agg(['median', 'min', 'max']).round(2).to_string())
top = g.loc[g.hedges_g.abs().idxmax()]
print(f'Largest |g|: {top.hedges_g:.2f} ({top.site}, {top.condition}; δ = {top.cliffs_delta:.2f})')

## 7. Individual values (Fig. S1)

Relative beta power (mean of the 19 channels and T4) and permutation entropy (mean of the 19
channels) of every participant, by condition, in MDD, January 2013 controls and
contemporaneous controls. Horizontal lines are medians.

In [ ]:
BLUE, ORANGE, TEAL = '#2a78d6', '#eb6834', '#1baf7a'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'font.family': 'serif', 'font.size': 7.5, 'pdf.fonttype': 42,
                     'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'axes.linewidth': 0.6,
                     'xtick.color': INK2, 'ytick.color': INK2, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
                     'xtick.labelcolor': INK, 'ytick.labelcolor': INK2})
GROUPS = {'MDD': MDD, 'HC_JAN': JAN, 'HC_CONT': CONTEMP}
COLOR = {'MDD': BLUE, 'HC_JAN': ORANGE, 'HC_CONT': TEAL}
MARKER = {'MDD': 'o', 'HC_JAN': 's', 'HC_CONT': '^'}
TICK = {'MDD': 'MDD', 'HC_JAN': 'HC\nJan', 'HC_CONT': 'HC\nMay–Jul'}
LEGEND = {'MDD': f'MDD (n = {len(MDD)})', 'HC_JAN': f'HC, Jan 2013 (n = {len(JAN)})',
          'HC_CONT': f'HC, May–Jul 2013 (n = {len(CONTEMP)})'}
ROWS = [('Relative beta power\n(mean of 19 channels)', lambda d: d[[f'rel_beta_{c}' for c in CHANNEL_NAMES]].mean(axis=1)),
        ('Relative beta power\n(T4)', lambda d: d['rel_beta_T4']),
        ('Permutation entropy\n(mean of 19 channels)', lambda d: d[[f'permen_{c}' for c in CHANNEL_NAMES]].mean(axis=1))]
signed = lambda d: f'{d:+.2f}'.replace('-', '−')

fig, axes = plt.subplots(3, 3, figsize=(7.16, 7.0), sharey='row')
rng = np.random.default_rng(SEED)
deltas = []
for j, cond in enumerate(CONDITIONS):
    for i, (label, fn) in enumerate(ROWS):
        v = fn(features[cond])
        ax = axes[i, j]
        for x, k in enumerate(GROUPS):
            y = v[GROUPS[k]].to_numpy()
            jitter = rng.uniform(-0.17, 0.17, len(y)) if len(y) > 3 else np.linspace(-0.1, 0.1, len(y))
            ax.scatter(x + jitter, y, s=22, marker=MARKER[k], color=COLOR[k], edgecolor='white', linewidth=0.6,
                       zorder=3, label=LEGEND[k])
            ax.hlines(np.median(y), x - 0.3, x + 0.3, color=INK, linewidth=1.4, zorder=4, capstyle='round')
        d_all, d_cont, d_batch = cliff(v[MDD], v[HC]), cliff(v[MDD], v[CONTEMP]), cliff(v[JAN], v[CONTEMP])
        deltas.append(dict(row='abc'[i], variable=label.replace('\n', ' '), condition=cond,
                           delta_mdd_all_hc=round(d_all, 2), delta_mdd_contemp=round(d_cont, 2),
                           delta_batch_jan_contemp=round(d_batch, 2)))
        ax.text(0.03, 0.97, f'δ vs. all HC: {signed(d_all)}\nδ vs. May–Jul HC: {signed(d_cont)}',
                transform=ax.transAxes, va='top', ha='left', fontsize=6.3, color=INK2, linespacing=1.35)
        ax.set_xticks(range(3))
        ax.set_xticklabels([TICK[k] for k in GROUPS] if i == 2 else [])
        ax.set_xlim(-0.6, 2.6)
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', color=GRID, linewidth=0.5, zorder=0)
        ax.set_axisbelow(True)
        if i == 0:
            ax.set_title(cond, fontsize=8.5, color=INK, pad=6)
        if j == 0:
            ax.set_ylabel(label)
            ax.text(-0.34, 1.04, f"({'abc'[i]})", transform=ax.transAxes, fontsize=8.5, fontweight='bold', color=INK)
for i in range(3):                       # room above the data for the annotations
    lo, hi = axes[i, 0].get_ylim()
    axes[i, 0].set_ylim(lo, hi + 0.30 * (hi - lo))
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0),
           fontsize=7.5, handletextpad=0.3, columnspacing=1.6)
fig.tight_layout(rect=(0.025, 0, 1, 0.965), h_pad=1.2, w_pad=0.8)
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join(OUT_DIR, f'figS1_individual_values.{ext}'), dpi=300)
plt.show()
deltas = pd.DataFrame(deltas)
deltas.to_csv(os.path.join(OUT_DIR, 'figS1_deltas.csv'), index=False)
print(deltas.to_string(index=False))

## 8. Connectivity and relative theta: features cited in the Results

Significant features (Layer 1) of alpha connectivity and relative theta, and the batch
analysis of the significant connectivity features (`batch_effect_by_feature.csv` of notebook 03)
by band and direction of the group difference.

In [ ]:
l1 = pd.read_csv(os.path.join(RESULTS_DIR, 'layer1_summary.csv'), float_precision='round_trip')
cols = ['feature'] + [f'detected_{c}' for c in CONDITIONS] + [f'cliffs_delta_{c}' for c in CONDITIONS]
for title, sel in (('Alpha connectivity', l1.feature.str.match(r'^(coh|plv)_.*_alpha$')),
                   ('Relative theta', l1.feature.str.startswith('rel_theta_'))):
    d = l1[sel & (l1.n_conditions_detected > 0)]
    print(f'{title}: significant in at least one condition: {len(d)}')
    for c in CONDITIONS:
        s = d[d[f'detected_{c}']]
        print(f"  {c:5s} {len(s):2d} significant (MDD > HC: {int((s[f'cliffs_delta_{c}'] > 0).sum())}, "
              f"MDD < HC: {int((s[f'cliffs_delta_{c}'] < 0).sum())})")
    print(d[cols].round(2).to_string(index=False) if len(d) else '  none', '\n')

batch = pd.read_csv(os.path.join(RESULTS_DIR, 'batch_effect_by_feature.csv'))
conn = batch[batch.feature.str.startswith(('coh_', 'plv_'))].copy()
conn['band'] = conn.feature.str.split('_').str[-1]
conn['direction'] = np.where(conn.delta_group > 0, 'MDD > HC', 'MDD < HC')
print('Significant connectivity features: median Cliff\'s δ by condition, band and direction')
print(conn.groupby(['condition', 'band', 'direction'])
      .agg(n=('feature', 'size'), delta_group=('delta_group', 'median'),
           delta_mdd_vs_contemp=('delta_mdd_vs_hc_other', 'median'), delta_batch=('delta_batch', 'median'))
      .round(2).to_string())